# 🔍 Knee MRI Discrepancy Auditor & Label Noise Inspector

This notebook performs two essential data-quality audits:
1. **Gold vs. Silver Benchmark (58 Expert Studies):** Evaluates extracted NLP/LLM labels against the 58 consensus gold standard cases to diagnose extraction false negatives/positives across 12 languages.
2. **Vision Model Out-of-Fold (OOF) vs. Silver Labels (Confident Learning):** Identifies silent findings (omissions in reports) and phantom findings (historical injuries misparsed) in the 4,349 unannotated training studies.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add repository root to path
root_dir = os.path.abspath("..")
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)

from scripts.label_mining.discrepancy_auditor import DiscrepancyAuditor
from scripts.label_mining.regex_labeler import TARGET_COLS

print("Discrepancy Auditor loaded successfully.")

## 1. Audit #1: Silver Extracted Labels vs. 58 Gold Standard Cases

In [ ]:
auditor = DiscrepancyAuditor(
    raw_train_path="../data/raw/train.csv",
    silver_labels_path="../data/processed/train_labels.csv"
)

summary_df, merged_df = auditor.audit_silver_vs_gold()
summary_df

### Visualizing Performance Across the 12 Targets

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=summary_df, x="Target", y="AUC", palette="viridis")
plt.xticks(rotation=45, ha="right")
plt.title("Silver Label Miner vs. 58 Gold Standard Cases (AUC ROC)")
plt.ylim(0.5, 1.0)
plt.axhline(0.85, color="red", linestyle="--", label="Target Quality Threshold (0.85)")
plt.legend()
plt.tight_layout()
plt.show()

## 2. False Negative Diagnostics (Finding What the Miner Missed)

In [ ]:
# Example: Inspect missed Medial Meniscus tears
target_to_inspect = "Medial Meniscus"
fn_cases = auditor.find_high_discrepancy_cases(merged_df, target=target_to_inspect, error_type="FN", top_n=5)

print(f"Found {len(fn_cases)} False Negative cases for {target_to_inspect}:")
for idx, row in fn_cases.iterrows():
    print(f"\n--- Study UID: {row['StudyInstanceUID']} ---")
    print(f"Gold: {row[f'{target_to_inspect}_gold']} | Silver Extracted: {row[f'{target_to_inspect}_silver']}")
    print("Report Excerpt:")
    print(row['Report'][:400])

## 3. Generate Noise-Robust Sample Weights for Model Training

In [ ]:
weighted_df = auditor.generate_confidence_sample_weights()
print("Sample weights distribution:")
print(weighted_df["sample_weight"].value_counts().sort_index(ascending=False))

# Save noise-robust training labels
weighted_df.to_csv("../data/processed/train_labels_weighted.csv", index=False)
print("\nSaved data/processed/train_labels_weighted.csv with sample weights.")

## 4. Audit #2: Vision Model Out-of-Fold (OOF) vs. Silver Labels

When model training completes in `train.py`, load `oof_predictions.csv` below to detect silent findings and phantom report hallucinations.

In [ ]:
oof_path = "../checkpoints/oof_predictions.csv"
if os.path.exists(oof_path):
    oof_df = pd.read_csv(oof_path)
    oof_discrepancies = auditor.audit_model_oof_vs_silver(oof_df)
    print(f"Total High-Discrepancy Samples Detected: {len(oof_discrepancies)}")
    display(oof_discrepancies.head(10))
else:
    print("No oof_predictions.csv found yet. Run train.py with 5-fold CV to generate OOF predictions for full Confident Learning analysis.")